In [ ]:
# ============================================================
# 1. Install packages, import libraries, initialize settings
# ============================================================

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

get_ipython().system('pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1')
get_ipython().system('pip -q uninstall -y ibis-framework')
get_ipython().system('pip -q install -U "toolz>=1.0.0"')

import re, gc, json, gzip, zipfile, traceback, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.special import softmax

from google.colab import drive
from tqdm.notebook import tqdm

from colabdesign.mpnn import mk_mpnn_model
from colabdesign.mpnn.model import residue_constants
from Bio.PDB import MMCIFParser, PDBIO, Select
import jax

print("All packages imported.")
print("AA order:", "".join(residue_constants.restypes))


In [ ]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

try:
    drive.flush_and_unmount()
    print("Old Drive mount flushed.")
except Exception:
    print("No previous Drive mount to flush.")

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
if DRIVE_ROOT.exists():
    print("Drive mounted:", DRIVE_ROOT)
else:
    raise RuntimeError("Drive mount failed.")

In [ ]:
# ============================================================
# 3. Project paths
# ============================================================

selectPDB = "selectPDB"          # <-- folder name inside raw/

PROJECT_DIR  = Path("/content/drive/MyDrive/Colab Notebook/PRS-MPNN")
RAW_DIR      = PROJECT_DIR / "raw"
INPUT_DIR    = RAW_DIR / selectPDB
RESULTS_ROOT = PROJECT_DIR / "results" / "MPNN"
MODE_DIR     = RESULTS_ROOT / "response" / selectPDB
TMP_ROOT     = Path("/content/tmp_alphafold_pdbs_response")
TARGET_CHAIN = "A"

for d in [RAW_DIR, RESULTS_ROOT, MODE_DIR, TMP_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("INPUT_DIR   :", INPUT_DIR)
print("MODE_DIR    :", MODE_DIR)


In [ ]:
# ============================================================
# 4. Runtime configuration  (edit these before running)
# ============================================================

RANDOM_SEED = 45
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
jax_rng_key = jax.random.PRNGKey(RANDOM_SEED)
print(f"Seeds fixed: Python/NumPy={RANDOM_SEED}, JAX key initialised.")

MODEL_NAME = "v_48_020"
RESPONSE_DEFINITION = "max_delta_probability_energy_gap_from_masked_probability_tensor"

MAX_PROTEINS   = None
MAX_LENGTH     = None
SKIP_FINISHED  = False
RESUME_PARTIAL = False
SAVE_EVERY_POS = 10

EPS_PROB = 1e-9

AA_LIST      = list(residue_constants.restypes)
TARGET_CHAIN = "A"

THREE_TO_ONE = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M",
}

print("Mode         : xscan tensor generation only")
print(f"R definition : {RESPONSE_DEFINITION}")
print(f"MODEL_NAME   : {MODEL_NAME}")
print(f"MAX_PROTEINS : {MAX_PROTEINS}")
print(f"INPUT_DIR    : {INPUT_DIR}")


In [ ]:
# ============================================================
# 5. Initialize ProteinMPNN
# ============================================================

mpnn_model = mk_mpnn_model(MODEL_NAME)
print(f"ProteinMPNN loaded: {MODEL_NAME}")

In [ ]:
# ============================================================
# 6. File discovery and CIF/ZIP handling
# ============================================================


def get_accession_from_filename(filepath):
    name = Path(filepath).name
    m = re.match(r"AF-(.+?)-F\d+-model_v\d+\.pdb(?:\.gz)?$", name)
    if m:
        return m.group(1)
    for ext in (".pdb.gz", ".pdb", ".zip", ".cif.gz", ".cif"):
        if name.lower().endswith(ext):
            return name[: -len(ext)]
    return Path(filepath).stem


def list_protein_files(input_dir):
    found = []
    for p in sorted(input_dir.iterdir()):
        if not p.is_file():
            continue
        low = p.name.lower()
        if low.endswith(".pdb") or low.endswith(".pdb.gz") or low.endswith(".zip"):
            found.append(p)
    return found


# ── CIF -> PDB via Biopython (replaces buggy hand-written parser) ──
# Biopython's MMCIFParser + PDBIO correctly handles insertion codes,
# chain naming, alt conformations, and column alignment.

class _ProteinChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    def accept_residue(self, residue):
        return residue.id[0] == " "


def convert_cif_to_pdb(cif_path, out_pdb_path, target_chain="A"):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("af3", str(cif_path))
    model = structure[0]

    chain_ids = [c.id for c in model.get_chains()]
    print(f"  CIF chains: {chain_ids}")

    if target_chain in chain_ids:
        use_chain = target_chain
    elif chain_ids:
        use_chain = chain_ids[0]
        print(f"  '{target_chain}' not found, using '{use_chain}'")
        model[use_chain].id = target_chain
    else:
        raise ValueError(f"No chains in {cif_path}")

    io = PDBIO()
    io.set_structure(model)
    io.save(str(out_pdb_path), select=_ProteinChainSelect(target_chain))

    n_res = sum(1 for r in model[target_chain] if r.id[0] == " ")
    print(f"  CIF -> PDB: {n_res} residues, {Path(out_pdb_path).name}")
    if n_res == 0:
        raise ValueError(f"0 residues written from {Path(cif_path).name}")


# ── AlphaFold3 ZIP handling ──

def _find_sequence_in_json(data):
    if isinstance(data, dict):
        if "sequence" in data and isinstance(data["sequence"], str):
            seq = data["sequence"]
            if len(seq) > 5 and all(c in "ARNDCQEGHILKMFPSTWYVX" for c in seq.upper()):
                return seq
        for v in data.values():
            r = _find_sequence_in_json(v)
            if r:
                return r
    if isinstance(data, list):
        for item in data:
            r = _find_sequence_in_json(item)
            if r:
                return r
    return None


def extract_zip_protein(zip_path, tmp_root, target_chain="A"):
    tmp_root.mkdir(parents=True, exist_ok=True)
    accession = zip_path.stem

    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        cif_name  = next((n for n in names if n.endswith("model_0.cif")), None)
        json_name = next((n for n in names if n.lower().endswith("job_request.json")), None)

        if not cif_name:
            raise FileNotFoundError(f"No *model_0.cif in {zip_path.name}")
        if not json_name:
            raise FileNotFoundError(f"No *job_request.json in {zip_path.name}")

        tmp_cif = tmp_root / f"{accession}_model_0.cif"
        tmp_cif.write_bytes(zf.read(cif_name))
        tmp_pdb = tmp_root / f"{accession}.pdb"
        convert_cif_to_pdb(tmp_cif, tmp_pdb, target_chain=target_chain)
        tmp_cif.unlink()

        job_data = json.loads(zf.read(json_name))
        sequence = _find_sequence_in_json(job_data)
        if not sequence:
            raise ValueError(f"No protein sequence in {json_name}")

    return tmp_pdb, sequence, accession


print("File utilities ready.")

In [ ]:
# ============================================================
# 7. PDB sequence reader
# ============================================================


def read_pdb_sequence(pdb_file, chain_id="A"):
    residues, seen = [], set()
    with open(pdb_file) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            if line[12:16].strip() != "CA":
                continue
            if line[21].strip() != chain_id:
                continue
            key = (line[21], line[22:26].strip(), line[26].strip())
            if key in seen:
                continue
            seen.add(key)
            residues.append(THREE_TO_ONE.get(line[17:20].strip(), "X"))
    seq = "".join(residues)
    if not seq:
        raise ValueError(f"No sequence for chain {chain_id} in {pdb_file}")
    return seq


def clean_sequence_for_mpnn(seq):
    invalid = sorted(set(seq) - set(AA_LIST))
    if invalid:
        raise ValueError(f"Unsupported residues: {invalid}")
    return seq

In [ ]:
# ============================================================
# 8. ProteinMPNN probability extraction
# ============================================================


def get_mpnn_probs(mpnn_model, seq, mode="conditional"):
    L = sum(mpnn_model._lengths)
    if mode == "unconditional":
        ar_mask = np.zeros((L, L), dtype=np.float32)
    elif mode == "conditional":
        ar_mask = 1 - np.eye(L, dtype=np.float32)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    result = mpnn_model.score(seq=seq, ar_mask=ar_mask)
    logits = np.asarray(result["logits"]).squeeze()
    if logits.ndim == 3:
        logits = logits[0]
    if logits.shape[0] != L and logits.shape[1] == L:
        logits = logits.T
    if logits.shape[0] != L:
        raise ValueError(f"Logits shape mismatch: {logits.shape}, L={L}")
    return softmax(logits[:, :20], axis=-1).astype(np.float32)

In [ ]:
# ============================================================
# 9. Core tensor generation functions
# ============================================================


def get_masked_source_probs(mpnn_model, wt_seq, source_j):
    """
    Return P_j_rm, an L x 20 probability matrix after masking source position j.
    This is the same masked forward pass used by the original X-scan routine.
    """
    L = len(wt_seq)
    ar_mask_j = 1 - np.eye(L, dtype=np.float32)
    ar_mask_j[:, source_j] = 0.0

    result = mpnn_model.score(seq=wt_seq, ar_mask=ar_mask_j)
    logits = np.asarray(result["logits"]).squeeze()
    if logits.ndim == 3:
        logits = logits[0]
    if logits.shape[0] != L and logits.shape[1] == L:
        logits = logits.T
    if logits.shape[0] != L:
        raise ValueError(f"Logits shape mismatch: {logits.shape}, L={L}")
    return softmax(logits[:, :20], axis=-1).astype(np.float32)


def compute_xscan_probability_tensor(mpnn_model, wt_seq, save_every=10):
    """
    Build d_tensor[i, j, a] = P_j_rm[i, a].

    Axis convention:
      i: receiver residue position
      j: masked source residue position
      a: amino-acid index in AA_LIST
    """
    L = len(wt_seq)
    d_tensor = np.zeros((L, L, 20), dtype=np.float32)

    with tqdm(total=L, desc="X-scan probability tensor", leave=False) as pbar:
        for j in range(L):
            d_tensor[:, j, :] = get_masked_source_probs(mpnn_model, wt_seq, j)
            pbar.update(1)
            if save_every and (j + 1) % save_every == 0:
                print(f"  computed masked source positions: {j + 1}/{L}")

    return d_tensor


In [ ]:
# ============================================================
# 10. Site metrics removed
# ============================================================

# This notebook now only generates ProteinMPNN probability tensors.
# R_xscan and site-level metrics are computed by the reusable post-processing
# script in test/unify/d_tensor_to_r_xscan.py.


In [ ]:
# ============================================================
# 11. Plotting removed
# ============================================================

# No figures are generated by this notebook.


In [ ]:
# ============================================================
# 12. Single-protein xscan tensor runner
# ============================================================


def run_one_protein_response_matrix(
    protein_name, pdb_file, chain_id, output_dir, mpnn_model,
    override_seq=None, skip_finished=True, resume_partial=True,
    max_length=None, save_every=10,
):
    protein_dir = output_dir / protein_name
    protein_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = protein_dir / f"{protein_name}_metadata.json"
    npz_path      = protein_dir / f"{protein_name}_xscan_probability_tensors.npz"
    required = [metadata_path, npz_path]

    if skip_finished and all(p.exists() for p in required):
        print(f"  [SKIP] {protein_name}: outputs exist.")
        return {"protein": protein_name, "status": "skipped_existing", "error": ""}

    # Sequence
    if override_seq is not None:
        wt_seq = clean_sequence_for_mpnn(override_seq)
        print(f"  Sequence from JSON: {len(wt_seq)} residues")
    else:
        wt_seq = clean_sequence_for_mpnn(read_pdb_sequence(pdb_file, chain_id))

    L = len(wt_seq)
    print(f"  Length: {L}  |  Mode: xscan tensor generation")
    if max_length and L > max_length:
        print(f"  [SKIP] L={L} > MAX_LENGTH={max_length}")
        return {"protein": protein_name, "status": "skipped_too_long", "length": L, "error": ""}

    wt_indices = np.asarray([AA_LIST.index(aa) for aa in wt_seq], dtype=np.int32)

    # Load structure
    mpnn_model.prep_inputs(
        pdb_filename=str(pdb_file), chain=chain_id,
        homooligomer=False, fix_pos=None, inverse=True, verbose=False,
    )

    print("  Computing P_wt conditional baseline ...", end=" ", flush=True)
    P_wt = get_mpnn_probs(mpnn_model, wt_seq, mode="conditional")
    print("done.")

    print("  Computing P_uncond unconditional baseline ...", end=" ", flush=True)
    P_uncond = get_mpnn_probs(mpnn_model, wt_seq, mode="unconditional")
    print("done.")

    print(f"  Computing d_tensor_LxLx20 from {L} masked source positions ...")
    d_tensor = compute_xscan_probability_tensor(
        mpnn_model=mpnn_model,
        wt_seq=wt_seq,
        save_every=save_every,
    )

    np.savez_compressed(
        npz_path,
        P_wt_Lx20=P_wt,
        P_uncond_Lx20=P_uncond,
        d_tensor_LxLx20=d_tensor,
        wt_indices=wt_indices,
        wt_sequence=np.asarray(wt_seq),
        aa_order=np.asarray("".join(AA_LIST)),
    )

    meta = {
        "protein": protein_name,
        "chain_id": chain_id,
        "length": L,
        "wt_sequence": wt_seq,
        "model_name": MODEL_NAME,
        "mode": "xscan_probability_tensor_generation",
        "response_definition": RESPONSE_DEFINITION,
        "random_seed": RANDOM_SEED,
        "aa_order": "".join(AA_LIST),
        "eps_prob": EPS_PROB,
        "n_forward_passes": L + 2,
        "tensor_definition": "d_tensor_LxLx20[i, j, a] = P_j_rm[i, a], where j is the masked source position",
        "npz_file": str(npz_path),
        "npz_arrays": ["P_wt_Lx20", "P_uncond_Lx20", "d_tensor_LxLx20", "wt_indices", "wt_sequence", "aa_order"],
        "timestamp_utc": datetime.utcnow().isoformat(),
    }
    with open(metadata_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"  Saved metadata: {metadata_path.name}")
    print(f"  Saved tensors : {npz_path.name}")
    print(f"  [DONE] {protein_name}  ({L + 2} forward passes)")

    return {
        "protein": protein_name,
        "status": "finished",
        "length": L,
        "n_forward_passes": L + 2,
        "metadata_path": str(metadata_path),
        "npz_path": str(npz_path),
        "error": "",
    }


In [ ]:
# ============================================================
# 13. Batch scan proteins from INPUT_DIR
# ============================================================

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")

protein_files  = list_protein_files(INPUT_DIR)
members_to_run = protein_files if MAX_PROTEINS is None else protein_files[:MAX_PROTEINS]

print(f"Found {len(protein_files)} protein files in '{INPUT_DIR.name}'.")
print(f"Will process {len(members_to_run)} proteins.")
print(f"Output: {MODE_DIR}")

batch_logs = []

for idx, fpath in enumerate(tqdm(members_to_run, desc="Proteins"), start=1):
    accession = get_accession_from_filename(fpath)
    file_type = ("zip" if fpath.name.lower().endswith(".zip") else
                 "pdb.gz" if fpath.name.lower().endswith(".pdb.gz") else "pdb")
    print(f"\n[{idx}/{len(members_to_run)}] {accession}  ({file_type})")

    tmp_pdb, override_seq = None, None

    try:
        if file_type == "zip":
            tmp_pdb, override_seq, accession = extract_zip_protein(
                fpath, TMP_ROOT, target_chain=TARGET_CHAIN)
        elif file_type == "pdb.gz":
            tmp_pdb = TMP_ROOT / f"{accession}.pdb"
            tmp_pdb.write_bytes(gzip.decompress(fpath.read_bytes()))
        else:
            tmp_pdb = fpath

        log = run_one_protein_response_matrix(
            protein_name=accession, pdb_file=tmp_pdb, chain_id=TARGET_CHAIN,
            output_dir=MODE_DIR, mpnn_model=mpnn_model, override_seq=override_seq,
            skip_finished=SKIP_FINISHED, resume_partial=RESUME_PARTIAL,
            max_length=MAX_LENGTH, save_every=SAVE_EVERY_POS)
        batch_logs.append(log)

    except Exception as e:
        print(f"  [ERROR] {accession}: {e}")
        traceback.print_exc()
        batch_logs.append({"protein": accession, "status": "failed", "error": str(e)})

    finally:
        if tmp_pdb is not None and tmp_pdb != fpath and tmp_pdb.exists():
            tmp_pdb.unlink()
        gc.collect()

log_df = pd.DataFrame(batch_logs)
print("\nBatch finished.")
display(log_df)


In [ ]:
# ============================================================
# 14. No merged site metrics
# ============================================================

print("Site metrics are not generated in this tensor-only workflow.")


In [ ]:
# ============================================================
# 15. Inspect a single protein's outputs
# ============================================================

INSPECT_PROTEIN = None

result_dirs = [d for d in sorted(MODE_DIR.iterdir()) if d.is_dir()]

if not result_dirs:
    print("No result directories found.")
else:
    target_dir = (MODE_DIR / INSPECT_PROTEIN) if INSPECT_PROTEIN else result_dirs[0]
    pname = target_dir.name
    print(f"Inspecting: {pname}")

    with open(target_dir / f"{pname}_metadata.json") as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2))

    npz_path = target_dir / f"{pname}_xscan_probability_tensors.npz"
    with np.load(npz_path) as data:
        for key in data.files:
            arr = data[key]
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")


In [ ]:
# ============================================================
# 16. List all output files
# ============================================================

print("Output root:", MODE_DIR)
for d in sorted(MODE_DIR.iterdir()):
    if not d.is_dir():
        continue
    print(f"\n  [{d.name}]")
    for f in sorted(d.rglob("*")):
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            print(f"    {str(f.relative_to(d)):<55s}  {size_kb:>8.1f} KB")
